# SentinelLM v1.0 — Autonomous Security AI
**Author:** @who_is_the_black_hat

**Architecture:** GPT-style Decoder-only Transformer
- 117M parameters
- 6 layers, 8 heads, 512 dim
- 512 context length
- BPE tokenizer 32K vocab

**Upload to Drive:**
- `sentinellm_train_v1.jsonl`

**After training copy to Kali:**
```bash
cp ~/Downloads/sentinellm_v1.pt      /home/kali/osints/models/ml_engine/
cp ~/Downloads/sentinellm_vocab.json /home/kali/osints/models/ml_engine/
```

In [ ]:
# Cell 1 — GPU Check
import torch
print('GPU  :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NOT AVAILABLE')
print('CUDA :', torch.cuda.is_available())
print('RAM  :', torch.cuda.get_device_properties(0).total_memory // 1024**3, 'GB')
assert torch.cuda.is_available(), 'GPU nahi mila!'
DEVICE = torch.device('cuda')

In [ ]:
# Cell 2 — Install + Mount Drive + Load Data
!pip install tokenizers -q
from google.colab import drive
import json, glob, random
from collections import Counter
random.seed(42)

drive.mount("/content/drive", force_remount=True)

def find_file(fname):
    found = glob.glob(f"/content/drive/MyDrive/**/{fname}", recursive=True)
    if not found: found = glob.glob(f"/content/drive/MyDrive/{fname}")
    return found[0] if found else None

path = find_file("sentinellm_train_v1.jsonl")
assert path, "sentinellm_train_v1.jsonl not found in Drive!"

print("Loading data...")
samples = []
seen = set()
with open(path, encoding="utf-8", errors="replace") as f:
    for line in f:
        try:
            d = json.loads(line.strip())
            inst = d.get("instruction","").strip()
            out  = d.get("output","").strip()
            inp  = d.get("input","").strip()
            if len(inst) < 10 or len(out) < 5: continue
            key = inst[:60].lower()
            if key in seen: continue
            seen.add(key)
            # Format: instruction + input + output as one text
            if inp:
                text = f"### Instruction:\n{inst}\n\n### Input:\n{inp}\n\n### Response:\n{out}"
            else:
                text = f"### Instruction:\n{inst}\n\n### Response:\n{out}"
            samples.append(text)
        except: pass

random.shuffle(samples)
print(f"Loaded: {len(samples):,} samples")
print(f"Sample:\n{samples[0][:200]}")


In [ ]:
# Cell 3 — BPE Tokenizer Training
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import ByteLevel
from tokenizers.processors import TemplateProcessing
import re, json

VOCAB_SIZE = 32000
MAX_LEN    = 512

# Corpus file banao
print("Building BPE corpus...")
corpus_path = "/tmp/sentinellm_corpus.txt"
with open(corpus_path, "w") as f:
    for text in samples:
        f.write(text.replace("\n", " ") + "\n")

# BPE train karo
tokenizer = Tokenizer(BPE(unk_token="<unk>"))
tokenizer.pre_tokenizer = ByteLevel(add_prefix_space=False)
trainer = BpeTrainer(
    vocab_size=VOCAB_SIZE,
    special_tokens=["<pad>","<unk>","<bos>","<eos>","<sep>"],
    min_frequency=2,
    show_progress=True,
)
tokenizer.train([corpus_path], trainer)

PAD_ID = tokenizer.token_to_id("<pad>")
BOS_ID = tokenizer.token_to_id("<bos>")
EOS_ID = tokenizer.token_to_id("<eos>")
UNK_ID = tokenizer.token_to_id("<unk>")

tokenizer.post_processor = TemplateProcessing(
    single="<bos> $A <eos>",
    special_tokens=[("<bos>", BOS_ID), ("<eos>", EOS_ID)],
)

tokenizer.save("/tmp/sentinellm_vocab.json")
actual_vocab = tokenizer.get_vocab_size()
print(f"Vocab size: {actual_vocab:,}")
print(f"PAD={PAD_ID} BOS={BOS_ID} EOS={EOS_ID}")

# Test
enc = tokenizer.encode("nmap -sV -sC target.com")
print(f"Test encode: {enc.tokens[:10]}")


In [ ]:
# Cell 4 — SentinelLM Architecture (GPT-style Decoder)
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

# Hyperparams — T4 pe 2-3 ghante mein train hoga
N_LAYERS  = 6
N_HEADS   = 8
D_MODEL   = 512
D_FF      = 2048
DROPOUT   = 0.1
CTX_LEN   = 512

class SentinelAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.n_heads  = N_HEADS
        self.d_head   = D_MODEL // N_HEADS
        self.qkv      = nn.Linear(D_MODEL, 3 * D_MODEL, bias=False)
        self.out_proj = nn.Linear(D_MODEL, D_MODEL, bias=False)
        self.drop     = nn.Dropout(DROPOUT)

    def forward(self, x, mask=None):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(D_MODEL, dim=2)
        q = q.view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        scale = math.sqrt(self.d_head)
        att = (q @ k.transpose(-2, -1)) / scale
        if mask is not None:
            att = att.masked_fill(mask[:, :, :T, :T] == 0, float("-inf"))
        att = self.drop(F.softmax(att, dim=-1))
        out = (att @ v).transpose(1, 2).contiguous().view(B, T, C)
        return self.out_proj(out)


class SentinelBlock(nn.Module):
    def __init__(self):
        super().__init__()
        self.ln1  = nn.LayerNorm(D_MODEL)
        self.attn = SentinelAttention()
        self.ln2  = nn.LayerNorm(D_MODEL)
        self.ff   = nn.Sequential(
            nn.Linear(D_MODEL, D_FF),
            nn.GELU(),
            nn.Dropout(DROPOUT),
            nn.Linear(D_FF, D_MODEL),
            nn.Dropout(DROPOUT),
        )

    def forward(self, x, mask=None):
        x = x + self.attn(self.ln1(x), mask)
        x = x + self.ff(self.ln2(x))
        return x


class SentinelLM(nn.Module):
    VERSION = "1.0"
    AUTHOR  = "who_is_the_black_hat"

    def __init__(self, vocab_size):
        super().__init__()
        self.tok_emb  = nn.Embedding(vocab_size, D_MODEL)
        self.pos_emb  = nn.Embedding(CTX_LEN, D_MODEL)
        self.drop     = nn.Dropout(DROPOUT)
        self.blocks   = nn.ModuleList([SentinelBlock() for _ in range(N_LAYERS)])
        self.ln_final = nn.LayerNorm(D_MODEL)
        self.lm_head  = nn.Linear(D_MODEL, vocab_size, bias=False)
        # Weight tying
        self.lm_head.weight = self.tok_emb.weight
        # Causal mask
        mask = torch.tril(torch.ones(CTX_LEN, CTX_LEN)).view(1, 1, CTX_LEN, CTX_LEN)
        self.register_buffer("mask", mask)
        # Init weights
        self.apply(self._init)

    def _init(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, std=0.02)
            if m.bias is not None: nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        pos  = torch.arange(T, device=idx.device)
        x    = self.drop(self.tok_emb(idx) + self.pos_emb(pos))
        for block in self.blocks:
            x = block(x, self.mask)
        x      = self.ln_final(x)
        logits = self.lm_head(x)
        loss   = None
        if targets is not None:
            loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)),
                targets.view(-1),
                ignore_index=PAD_ID,
            )
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new=200, temperature=0.8, top_k=50):
        self.eval()
        for _ in range(max_new):
            idx_cond = idx[:, -CTX_LEN:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            logits[:, PAD_ID] = float("-inf")
            logits[:, UNK_ID] = float("-inf")
            if top_k > 0:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = float("-inf")
            probs  = F.softmax(logits, dim=-1)
            nxt    = torch.multinomial(probs, 1)
            if nxt.item() == EOS_ID: break
            idx = torch.cat([idx, nxt], dim=1)
        return idx

    def info(self):
        p = sum(x.numel() for x in self.parameters())
        return {"params": p, "size_mb": round(p*4/1024/1024, 2)}


model = SentinelLM(vocab_size=actual_vocab).to(DEVICE)
info  = model.info()
print(f"SentinelLM v1.0")
print(f"Params : {info['params']:,}")
print(f"Size   : {info['size_mb']} MB")
print(f"Layers : {N_LAYERS} | Heads: {N_HEADS} | Dim: {D_MODEL}")


In [ ]:
# Cell 5 — Dataset + Training
from torch.utils.data import Dataset, DataLoader
import time

BATCH_SIZE = 16
EPOCHS     = 2
LR         = 3e-4
GRAD_ACCUM = 4   # effective batch = 64
MAX_SAMPLES = 150000  # T4 RAM ke liye

class LMDataset(Dataset):
    def __init__(self, texts, max_len=CTX_LEN):
        self.data = []
        print("Tokenizing...")
        for i, text in enumerate(texts):
            enc = tokenizer.encode(text)
            ids = enc.ids[:max_len + 1]
            if len(ids) < 8: continue
            # Pad to max_len+1
            ids += [PAD_ID] * (max_len + 1 - len(ids))
            self.data.append(ids)
            if (i+1) % 50000 == 0:
                print(f"  {i+1:,} tokenized")
        print(f"Dataset: {len(self.data):,} samples")

    def __len__(self): return len(self.data)

    def __getitem__(self, idx):
        ids = self.data[idx]
        x = torch.tensor(ids[:-1], dtype=torch.long)
        y = torch.tensor(ids[1:],  dtype=torch.long)
        return x, y


# Split train/val
use_samples = samples[:MAX_SAMPLES]
n_val = max(500, int(len(use_samples) * 0.02))
val_texts   = use_samples[:n_val]
train_texts = use_samples[n_val:]
print(f"Train: {len(train_texts):,} | Val: {len(val_texts):,}")

train_ds = LMDataset(train_texts)
val_ds   = LMDataset(val_texts)

train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                      num_workers=2, pin_memory=True)
val_dl   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                      num_workers=2, pin_memory=True)

# Optimizer + Scheduler
optimizer = torch.optim.AdamW(model.parameters(), lr=LR,
                              weight_decay=0.1, betas=(0.9, 0.95))
total_steps = len(train_dl) * EPOCHS // GRAD_ACCUM
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=total_steps, eta_min=LR * 0.1)

# Training loop
print(f"Training {EPOCHS} epochs | {len(train_dl):,} batches/epoch")
print(f"Effective batch size: {BATCH_SIZE * GRAD_ACCUM}")
print()

best_val_loss = float("inf")
best_state    = None

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0
    optimizer.zero_grad()
    t0 = time.time()

    for step, (x, y) in enumerate(train_dl):
        x, y = x.to(DEVICE), y.to(DEVICE)
        _, loss = model(x, y)
        loss = loss / GRAD_ACCUM
        loss.backward()
        total_loss += loss.item() * GRAD_ACCUM

        if (step + 1) % GRAD_ACCUM == 0:
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

        if (step + 1) % 500 == 0:
            elapsed = time.time() - t0
            print(f"  Epoch {epoch} Step {step+1}/{len(train_dl)} | loss={total_loss/(step+1):.4f} | {elapsed:.0f}s")

    # Validation
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for x, y in val_dl:
            x, y = x.to(DEVICE), y.to(DEVICE)
            _, loss = model(x, y)
            val_loss += loss.item()
    val_loss /= len(val_dl)
    train_loss = total_loss / len(train_dl)

    print(f"Epoch {epoch} | train={train_loss:.4f} | val={val_loss:.4f} | time={time.time()-t0:.0f}s")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
        print(f"  Best model saved (val={val_loss:.4f})")

model.load_state_dict(best_state)
print(f"Training complete! Best val loss: {best_val_loss:.4f}")


In [ ]:
# Cell 6 — Save + Inference Test + Download
from google.colab import files
import shutil

# Save checkpoint
checkpoint = {
    "model_state": model.state_dict(),
    "config": {
        "vocab_size":  actual_vocab,
        "n_layers":    N_LAYERS,
        "n_heads":     N_HEADS,
        "d_model":     D_MODEL,
        "d_ff":        D_FF,
        "dropout":     DROPOUT,
        "ctx_len":     CTX_LEN,
    },
    "special_tokens": {
        "pad_id": PAD_ID,
        "bos_id": BOS_ID,
        "eos_id": EOS_ID,
        "unk_id": UNK_ID,
    },
    "version":      "1.0",
    "author":       "who_is_the_black_hat",
    "best_val_loss": best_val_loss,
    "params":       model.info()["params"],
    "size_mb":      model.info()["size_mb"],
    "saved_at":     __import__("time").strftime("%Y-%m-%d %H:%M:%S"),
    "train_samples": len(train_texts),
}
torch.save(checkpoint, "sentinellm_v1.pt")
shutil.copy("/tmp/sentinellm_vocab.json", "sentinellm_vocab.json")

print(f"Saved!")
print(f"Params    : {model.info()['params']:,}")
print(f"Size      : {model.info()['size_mb']} MB")
print(f"Val Loss  : {best_val_loss:.4f}")

# Inference test
def generate(prompt, max_new=150, temperature=0.8, top_k=50):
    model.eval()
    enc = tokenizer.encode(f"### Instruction:\n{prompt}\n\n### Response:\n")
    ids = torch.tensor([enc.ids[-CTX_LEN:]], dtype=torch.long).to(DEVICE)
    out = model.generate(ids, max_new=max_new, temperature=temperature, top_k=top_k)
    decoded = tokenizer.decode(out[0].tolist())
    # Response part extract karo
    if "### Response:" in decoded:
        return decoded.split("### Response:")[-1].strip()
    return decoded.strip()

tests = [
    "What is SQL injection and how to test for it?",
    "Generate nmap command for full port scan",
    "How to check for XSS vulnerability?",
    "What tool should I use after finding open port 80?",
]
print("\n=== Inference Test ===")
for t in tests:
    out = generate(t, max_new=100)
    print(f"Q: {t}")
    print(f"A: {out[:150]}")
    print()

# Download
files.download("sentinellm_v1.pt")
files.download("sentinellm_vocab.json")
print("Downloaded!")
print()
print("Kali pe copy karo:")
print("cp ~/Downloads/sentinellm_v1.pt      /home/kali/osints/models/ml_engine/")
print("cp ~/Downloads/sentinellm_vocab.json /home/kali/osints/models/ml_engine/")
